# Multiaxial Universal ML plasticity model

To generalise the formulation in [Metal1D](https://github.com/sinaplatform/plasticity-models/blob/main/Metal1D.ipynb) into a multiaxial case, first consider the elastic case as follow

$$ d\sigma= \mathbb{C}^e d\varepsilon^e$$

where in Matrix Form (Voigt Notation) $\mathbb{C}^e$ is 6 by 6 full matrix, it means that the stress increment in one direction is coupled with all loading directions.

$\textbf{IMPORTANT obeservation:}$ Accourding to the 1D formulation in [Metal1D](https://github.com/sinaplatform/plasticity-models/blob/main/Metal1D.ipynb), it seems that $\mathbb{C}$ itself if coupled with intenal state variables is able to handle history dependency!


Let us forget about the seperation of elastic and plastic part of material response and assume that the material is nonlinear under any loading magnitude. 

$$ d\sigma= \mathbb{C} d\varepsilon$$

Note that $\sigma$ and $\varepsilon$ are second order tensors in 3D and the material stiffness (tangental or Jacobian) $\mathbb{C}$ is a fourth-order tensor

$$
d \sigma_{ij} =  \mathbb{C}_{ijkl} d \varepsilon_{kl}
$$

where 

$$ 
\mathbb{C}_{ijkl} = \frac{\partial d \sigma_{ij}}{\partial d \varepsilon_{kl}}.
$$

In Matrix Form (Voigt Notation):

$$
d \sigma = 
\begin{bmatrix}
d \sigma_{11} \\
d \sigma_{22} \\
d \sigma_{33} \\
d \sigma_{23} \\
d \sigma_{31} \\
d \sigma_{12}
\end{bmatrix}
,~~~
d \varepsilon = 
\begin{bmatrix}
d \varepsilon_{11} \\
d \varepsilon_{22} \\
d \varepsilon_{33} \\
2 d \varepsilon_{23} \\
2 d \varepsilon_{31} \\
2 d \varepsilon_{12}
\end{bmatrix}
$$

These are different in ABAUQS.

$$
\mathbb{C} =
\begin{bmatrix}
C_{11} & C_{12} & C_{13} & C_{14} & C_{15} & C_{16} \\
C_{21} & C_{22} & C_{23} & C_{24} & C_{25} & C_{26} \\
C_{31} & C_{32} & C_{33} & C_{34} & C_{35} & C_{36} \\
C_{41} & C_{42} & C_{43} & C_{44} & C_{45} & C_{46} \\
C_{51} & C_{52} & C_{53} & C_{54} & C_{55} & C_{56} \\
C_{61} & C_{62} & C_{63} & C_{64} & C_{65} & C_{66} \\
\end{bmatrix}.
$$


<!-- Therefore a universal dynamic ML material plasticity model will look like this:

$$ \left\{
    \begin{aligned}
        d\sigma &= f(h) d\varepsilon\\
        dh &=g(h) d\varepsilon \\
    \end{aligned}
    \right. 
$$

or 

$$ \left\{
    \begin{aligned}
        \frac{d\sigma}{d\varepsilon}  &= f(h) \\
        \frac{dh}{d\varepsilon} &=g(h) \\
    \end{aligned}
    \right. 
$$

where $f$ and $g$ are neural operators (NO) with dimensions $f: \mathbb{R}^k \rightarrow \mathbb{R}^6 $ and $g: \mathbb{R}^k \rightarrow \mathbb{R}^6 $ for a 3D case. Here, $k$ is the number of states and we consider the Jacobian of states as below which for training purposes flattens.

$$
\frac{d\boldsymbol{h}}{d\boldsymbol{\varepsilon}} =
\begin{bmatrix}
\frac{\partial h^1}{\partial \varepsilon^1} & \frac{\partial h^1}{\partial \varepsilon^2} & \cdots \\
\frac{\partial h^2}{\partial \varepsilon^1} & \frac{\partial h^2}{\partial \varepsilon^2} & \cdots \\
\vdots & \vdots & \ddots
\end{bmatrix}
$$

Yet, we can use only one NN for all state evolutions or one NN for each state variable in all directions.  -->

## New state space formulation
Considering the [combined isotropic/kinematic modeling in 1D](https://github.com/sinaplatform/plasticity-models/blob/main/VonMisesMetal1D.ipynb)

$$ \left\{
    \begin{aligned}
        \frac{d\boldsymbol{\sigma}}{d\boldsymbol{\varepsilon}}  &= f(\boldsymbol{\sigma},\boldsymbol{h}) \\
        \frac{d\boldsymbol{h}}{d\boldsymbol{\varepsilon}} &=g(\boldsymbol{\sigma},\boldsymbol{h}) \\
    \end{aligned}
    \right. 
$$

if $ \boldsymbol{y} = \begin{bmatrix}
\boldsymbol{\sigma} \\
\boldsymbol{h}
\end{bmatrix} $ the universal state space ML model will be as follows:

$$ \left\{
    \begin{aligned}
        d\boldsymbol{y}(s) &= \mathscr{NO}(\boldsymbol{y}(s), \theta_{\boldsymbol{w},\boldsymbol{b}}) ~ d \boldsymbol{\varepsilon}(s) \\
        \boldsymbol{y}(0) & = \begin{bmatrix}
                    \boldsymbol{\sigma} (0) \\
                    \boldsymbol{h} (0)
                    \end{bmatrix} \\
        \hat{\boldsymbol{\sigma}} &= \boldsymbol{\sigma}(1)          
    \end{aligned}
    \right. \quad s \in[0,1]
$$


where $\mathscr{NO}$ is neural operators with dimensions $\mathscr{NO}: \mathbb{R}^{o+k} \rightarrow \mathbb{R}^{o(o+1)/2+k \times o} $. 
Here, $s$ is the increment step, and  $k$ is the number of states and $o$ is the number of obervable states which correspond to the stress components.
Note that the first $o$ raws in the output is a symmetric matrix and therefore the learnable output dimension is only $o(o+1)/2+k \times o$.


$$ 
\min_{\theta = \{w, b\}} \text{MSE} = \frac{1}{q \cdot m} \sum_{p=1}^{q} \sum_{n=1}^{m} \left(\sigma_p^n - \hat{\sigma}_p^n\right)^2
$$ 

This architecture directly learns material jacobian and jacobian of internal state variables ($h^k$).

$$
\frac{d\boldsymbol{y}}{d\boldsymbol{\varepsilon}} =
\begin{bmatrix}
\frac{\partial \sigma^1}{\partial \varepsilon^1} & \frac{\partial \sigma^1}{\partial \varepsilon^2} & \frac{\partial \sigma^1}{\partial \varepsilon^3} &
\frac{\partial \sigma^1}{\partial \varepsilon^4} &  \frac{\partial \sigma^1}{\partial \varepsilon^5} & \frac{\partial \sigma^1}{\partial \varepsilon^6} 
 \\
        & \frac{\partial \sigma^2}{\partial \varepsilon^2} & \frac{\partial \sigma^2}{\partial \varepsilon^3} &
\frac{\partial \sigma^2}{\partial \varepsilon^4} &  \frac{\partial \sigma^2}{\partial \varepsilon^5} & \frac{\partial \sigma^2}{\partial \varepsilon^6} 
 \\
     &   & \frac{\partial \sigma^3}{\partial \varepsilon^3} &
\frac{\partial \sigma^3}{\partial \varepsilon^4} &  \frac{\partial \sigma^3}{\partial \varepsilon^5} & \frac{\partial \sigma^3}{\partial \varepsilon^6} 
 \\
     &   &   &
\frac{\partial \sigma^4}{\partial \varepsilon^4} &  \frac{\partial \sigma^4}{\partial \varepsilon^5} & \frac{\partial \sigma^4}{\partial \varepsilon^6} 
 \\
     & \text{sym}  &   &
     &  \frac{\partial \sigma^5}{\partial \varepsilon^5} & \frac{\partial \sigma^5}{\partial \varepsilon^6} 
 \\
     &   &   &
     &       & \frac{\partial \sigma^6}{\partial \varepsilon^6} 
 \\ \hdashline

\frac{\partial h^1}{\partial \varepsilon^1} & \frac{\partial h^1}{\partial \varepsilon^2} & \frac{\partial h^1}{\partial \varepsilon^3} &
\frac{\partial h^1}{\partial \varepsilon^4} &  \frac{\partial h^1}{\partial \varepsilon^5} & \frac{\partial h^1}{\partial \varepsilon^6} 
 \\
\frac{\partial h^2}{\partial \varepsilon^1} & \frac{\partial h^2}{\partial \varepsilon^2} & \frac{\partial h^1}{\partial \varepsilon^3} &
\frac{\partial h^2}{\partial \varepsilon^4} &  \frac{\partial h^2}{\partial \varepsilon^5} & \frac{\partial h^2}{\partial \varepsilon^6}  \\
\vdots & \vdots & \vdots &\vdots &\vdots &\vdots 
\end{bmatrix}
$$


In [1]:
import os
import argparse
import time
from datetime import datetime
import numpy as np
import math

import torch
import torch.nn as nn
import torch.optim as optim

import torchdyn
from torchdyn.core import NeuralODE
from typing import Tuple

from torchdiffeq import odeint

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, MaxAbsScaler

import matplotlib.pyplot as plt
from ipywidgets import interact
%matplotlib qt
import pyqtgraph as pg
from pyqtgraph.Qt import QtGui
plt.rcdefaults()

# figure styling
font=16
font_axis=20
plt.rcParams.update({'font.size': font})  # Set the desired font size
# plt.rcParams['text.usetex'] = True
plt.rcParams['figure.facecolor'] = 'white'  # Background color for figures
plt.rcParams['axes.facecolor'] = 'white'    # Background color for axes

# Define the neural network model training parameters
args = argparse.Namespace(
      method='dopri5',
      data_size=200,
      batch_time=1,
      batch_size=20,
      niters=2000,
      test_freq=20,
      viz=True,
      gpu=0,
      adjoint=False
  )

if args.adjoint:
    from torchdiffeq import odeint_adjoint as odeint
else:
    from torchdiffeq import odeint

device = torch.device('cuda:' + str(args.gpu) if torch.cuda.is_available() else 'cpu')

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x00000236D7B06900>>
Traceback (most recent call last):
  File "c:\Users\lt24550\OneDrive - University of Bristol\SINDRI\Codes\SINDRIenv\Lib\site-packages\ipykernel\ipkernel.py", line 790, in _clean_thread_parent_frames
    active_threads = {thread.ident for thread in threading.enumerate()}
                                                 ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lt24550\AppData\Local\Programs\Python\Python312\Lib\threading.py", line 1533, in enumerate
    def enumerate():
    
KeyboardInterrupt: 


In [ ]:
# Record the start time
start_time = datetime.now()
print(f"Run started at: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

# import and navigate through the dataset 
time = np.expand_dims(np.load('t.npy'), axis=-1)
strain_full = np.expand_dims(np.load('strain.npy'), axis=-1)
stress_full = np.expand_dims(np.load('S.npy'), axis=-1)
state = np.expand_dims(np.load('state.npy'), axis=-1) # z in the paper

# Record the end time
end_time = datetime.now()
print(f"Run ended at: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")

# Optionally, print the duration
duration = end_time - start_time
print(f"Total run duration: {duration}")

In [ ]:
traj_ID = 0  # Trajectory ID to visualize

# Define component names corresponding to the 6 components
components = ['11', '22', '33', '12', '13', '23']

# Create a 2x3 grid of subplots to reflect tensor symmetry
fig, axs = plt.subplots(2, 3, figsize=(12, 7))

for i, ax in enumerate(axs.flat):
    if i < 6:
        component = components[i]
        strain = strain_full[traj_ID, :, i]  # Shape: (101,)
        stress = stress_full[traj_ID, :, i] / 1e6  # Convert to MPa if stress is in Pascals

        # Plot strain vs. stress
        ax.plot(strain, stress, 'k', label='Data')

        # Set labels and title
        ax.set_xlabel(f'$\epsilon_{{{component}}}$', fontsize=font, fontname='Times New Roman')
        ax.set_ylabel(f'$\sigma_{{{component}}}$', fontsize=font, fontname='Times New Roman')

        # Calculate maximum absolute values for symmetric axis limits
        max_strain = np.max(np.abs(strain))*1.1
        max_stress = np.max(np.abs(stress))*1.1

        # Set symmetric axis limits
        ax.set_xlim(-max_strain, max_strain)
        ax.set_ylim(-max_stress, max_stress)

        # Add legend only to the first subplot or as needed
        if i == 2:
            ax.legend(loc='upper right', prop={'family': 'Times New Roman', 'size': font})

        # Enable grid for better readability
        ax.grid(True)
    else:
        # Hide any unused subplots (if the grid has more subplots than components)
        ax.axis('off')

plt.tight_layout()
plt.show()

# plt.savefig('strain_stress_visualization.svg', dpi=600)

In [4]:
# Define the split percentages (60% training, 20% validation, 20% testing)
train_split = 0.001

# Number of samples
num_trajectory = strain_full.shape[0]

val_split = (1-train_split)/2
test_split = 1 - train_split - val_split


# Compute the split indices
train_idx = math.floor(num_trajectory * train_split)
val_idx = math.floor(num_trajectory * (train_split + val_split))

# F_full = torch.tensor(F_full, dtype=torch.float32)
strain_full = torch.tensor(strain_full, dtype=torch.float32)
stress_full = torch.tensor(stress_full, dtype=torch.float32)

strain_train = strain_full[:train_idx,:,0:1]
strain_val = strain_full[train_idx:val_idx,:,0:1]
strain_test = strain_full[val_idx:,:,0:1]

stress_train = stress_full[:train_idx,:,0:1]
stress_val = stress_full[train_idx:val_idx,:,0:1]
stress_test = stress_full[val_idx:,:,0:1]


def custom_scale(data):
    max_val = np.max(abs(data))
    scaled_data = data / max_val
    return scaled_data

# Scale the data
# scaler_E = MinMaxScaler(feature_range=(-1, 1), copy=True, clip=True)
# scaler_S = MinMaxScaler(feature_range=(-1, 1), copy=True, clip=True)

scaler_E = MaxAbsScaler()
scaler_S = MaxAbsScaler()

# strain_train_reshaped = strain_train.view(-1, strain_train.size(-1)).numpy()  # Reshape to 2D for scaling
# stress_train_reshaped = stress_train.view(-1, stress_train.size(-1)).numpy()
strain_train_reshaped=strain_train.reshape(-1, strain_train.size(-1))
stress_train_reshaped=stress_train.reshape(-1, stress_train.size(-1))
num_datasets=np.size(strain_train,0)
timesteps=np.size(strain_train,1)
strain_train_scaled = torch.tensor(scaler_E.fit_transform(strain_train_reshaped).reshape(num_datasets, timesteps, strain_train.size(-1)), dtype=torch.float32)
stress_train_scaled = torch.tensor(scaler_S.fit_transform(stress_train_reshaped).reshape(num_datasets, timesteps, stress_train.size(-1)), dtype=torch.float32)

# strain_val_reshaped = strain_val.view(-1, strain_val.size(-1)).numpy()  # Reshape to 2D for scaling
# stress_val_reshaped = stress_val.view(-1, stress_val.size(-1)).numpy()
strain_val_reshaped=strain_val.reshape(-1, strain_val.size(-1))
stress_val_reshaped=stress_val.reshape(-1, stress_val.size(-1))
num_datasets=np.size(strain_val,0)
timesteps=np.size(stress_val,1)
strain_val_scaled = torch.tensor(scaler_E.transform(strain_val_reshaped).reshape(num_datasets, timesteps, strain_val.size(-1)), dtype=torch.float32)
stress_val_scaled = torch.tensor(scaler_S.transform(stress_val_reshaped).reshape(num_datasets, timesteps, stress_val.size(-1)), dtype=torch.float32)

# strain_test_reshaped = strain_test.view(-1, strain_test.size(-1)).numpy()  # Reshape to 2D for scaling
# stress_test_reshaped = stress_test.view(-1, stress_test.size(-1)).numpy()
strain_test_reshaped=strain_test.reshape(-1, strain_test.size(-1))
stress_test_reshaped=stress_test.reshape(-1, stress_test.size(-1))
num_datasets=np.size(strain_test,0)
timesteps=np.size(strain_test,1)
strain_test_scaled = torch.tensor(scaler_E.transform(strain_test_reshaped).reshape(num_datasets, timesteps, strain_test.size(-1)), dtype=torch.float32)
stress_test_scaled = torch.tensor(scaler_S.transform(stress_test_reshaped).reshape(num_datasets, timesteps, stress_test.size(-1)), dtype=torch.float32)

# # Extract scaling parameters
# print(scaler_E.min_)
# scaling_params_strain = {
#     "min_": scaler_E.min_,           # min_ is used to scale data
#     "scale_": scaler_E.scale_,       # scale_ is the scaling factor
#     "data_min": scaler_E.data_min_, # original data min
#     "data_max": scaler_E.data_max_, # original data max
#     "data_range": scaler_E.data_range_ # data range (max - min)
# }

# # Save the scaling parameters to a .dat file
# with open('scaling_parameters_strain.dat', 'w') as file:
#     for key, value in scaling_params_strain.items():
#         file.write(f"{key}: {value.tolist()}\n")


# # Extract scaling parameters
# print(scaler_S.min_)
# scaling_params_stress = {
#     "min_": scaler_S.min_,           # min_ is used to scale data
#     "scale_": scaler_S.scale_,       # scale_ is the scaling factor
#     "data_min": scaler_S.data_min_, # original data min
#     "data_max": scaler_S.data_max_, # original data max
#     "data_range": scaler_S.data_range_ # data range (max - min)
# }

# # Save the scaling parameters to a .dat file
# with open('scaling_parameters_stress.dat', 'w') as file:
#     for key, value in scaling_params_stress.items():
#         file.write(f"{key}: {value.tolist()}\n")

# # see the scaling parameters work
# # to scale the data: data * scale + min
# # to reverse scale the data: (data - min) / scale

In [ ]:
traj_ID = 0  # Trajectory ID to visualize

# Define component names corresponding to the 6 components
components = ['11', '22', '33', '12', '13', '23']

# Create a 2x3 grid of subplots to reflect tensor symmetry
fig, axs = plt.subplots(2, 3, figsize=(12, 7))

for i, ax in enumerate(axs.flat):
    if i < 6:
        component = components[i]
        strain = strain_train_scaled[traj_ID, :, i].numpy()  # Shape: (101,)
        stress = stress_train_scaled[traj_ID, :, i].numpy() # Convert to MPa if stress is in Pascals

        # Plot strain vs. stress
        ax.plot(strain, stress, 'k', label='Data')

        # Set labels and title
        ax.set_xlabel(f'$\epsilon_{{{component}}}$', fontsize=font, fontname='Times New Roman')
        ax.set_ylabel(f'$\sigma_{{{component}}}$', fontsize=font, fontname='Times New Roman')

        # Calculate maximum absolute values for symmetric axis limits
        max_strain = np.max(np.abs(strain))*1.1
        max_stress = np.max(np.abs(stress))*1.1

        # Set symmetric axis limits
        ax.set_xlim(-max_strain, max_strain)
        ax.set_ylim(-max_stress, max_stress)

        # Add legend only to the first subplot or as needed
        if i == 2:
            ax.legend(loc='upper right', prop={'family': 'Times New Roman', 'size': font})

        # Enable grid for better readability
        ax.grid(True)
    else:
        # Hide any unused subplots (if the grid has more subplots than components)
        ax.axis('off')

plt.tight_layout()
plt.show()


# Reshape data to (1100*101, 6) for easier plotting of distributions
data_reshaped = strain_train_scaled.reshape(-1, 6)
#Plot the distribution for each component
fig, axs = plt.subplots(2, 3, figsize=(15, 10))

for i in range(6):
    row = i // 3
    col = i % 3
    axs[row, col].hist(data_reshaped[:, i], bins=30, color='blue', alpha=0.7)
    axs[row, col].set_title(f'Component {i+1} Distribution')
    axs[row, col].set_xlabel('Value')
    axs[row, col].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Reshape data to (1100*101, 6) for easier plotting of distributions
data_reshaped = stress_train_scaled.reshape(-1, 6)
#Plot the distribution for each component
fig, axs = plt.subplots(2, 3, figsize=(15, 10))

for i in range(6):
    row = i // 3
    col = i % 3
    axs[row, col].hist(data_reshaped[:, i], bins=30, color='blue', alpha=0.7)
    axs[row, col].set_title(f'Component {i+1} Distribution')
    axs[row, col].set_xlabel('Value')
    axs[row, col].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# plt.savefig('strain_stress_visualization.svg', dpi=600)

In [ ]:
def get_batch():
    s = torch.from_numpy(np.random.choice(np.arange(args.data_size - args.batch_time, dtype=np.int64), args.batch_size, replace=False))
    batch_y0 = true_y[s]  # (M, D)
    batch_t = t[:args.batch_time]  # (T)
    batch_y = torch.stack([true_y[s + i] for i in range(args.batch_time)], dim=0)  # (T, M, D)
    return batch_y0.to(device), batch_t.to(device), batch_y.to(device)


def makedirs(dirname):
    if not os.path.exists(dirname):
        os.makedirs(dirname)


if args.viz:
    makedirs('png')
    fig = plt.figure(figsize=(12, 4), facecolor='white')
    ax_traj = fig.add_subplot(131, frameon=False)
    ax_phase = fig.add_subplot(132, frameon=False)
    ax_vecfield = fig.add_subplot(133, frameon=False)
    plt.show(block=False)


def visualize(true_y, pred_y, odefunc, itr):

    if args.viz:

        ax_traj.cla()
        ax_traj.set_title('Trajectories')
        ax_traj.set_xlabel('t')
        ax_traj.set_ylabel('x,y')
        ax_traj.plot(t.cpu().numpy(), true_y.cpu().numpy()[:, 0, 0], t.cpu().numpy(), true_y.cpu().numpy()[:, 0, 1], 'g-')
        ax_traj.plot(t.cpu().numpy(), pred_y.cpu().numpy()[:, 0, 0], '--', t.cpu().numpy(), pred_y.cpu().numpy()[:, 0, 1], 'b--')
        ax_traj.set_xlim(t.cpu().min(), t.cpu().max())
        ax_traj.set_ylim(-2, 2)
        ax_traj.legend()

        ax_phase.cla()
        ax_phase.set_title('Phase Portrait')
        ax_phase.set_xlabel('x')
        ax_phase.set_ylabel('y')
        ax_phase.plot(true_y.cpu().numpy()[:, 0, 0], true_y.cpu().numpy()[:, 0, 1], 'g-')
        ax_phase.plot(pred_y.cpu().numpy()[:, 0, 0], pred_y.cpu().numpy()[:, 0, 1], 'b--')
        ax_phase.set_xlim(-2, 2)
        ax_phase.set_ylim(-2, 2)

        ax_vecfield.cla()
        ax_vecfield.set_title('Learned Vector Field')
        ax_vecfield.set_xlabel('x')
        ax_vecfield.set_ylabel('y')

        y, x = np.mgrid[-2:2:21j, -2:2:21j]
        dydt = odefunc(0, torch.Tensor(np.stack([x, y], -1).reshape(21 * 21, 2)).to(device)).cpu().detach().numpy()
        mag = np.sqrt(dydt[:, 0]**2 + dydt[:, 1]**2).reshape(-1, 1)
        dydt = (dydt / mag)
        dydt = dydt.reshape(21, 21, 2)

        ax_vecfield.streamplot(x, y, dydt[:, :, 0], dydt[:, :, 1], color="black")
        ax_vecfield.set_xlim(-2, 2)
        ax_vecfield.set_ylim(-2, 2)

        fig.tight_layout()
        plt.savefig('png/{:03d}'.format(itr))
        plt.draw()
        plt.pause(0.1)


class ODEFunc(nn.Module):

    def __init__(self):
        super(ODEFunc, self).__init__()

        self.net = nn.Sequential(
            nn.Linear(2, 100),
            nn.Tanh(),
            nn.Linear(100, 100),
            nn.Tanh(),
            nn.Linear(100, 2),
        )

        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0, std=0.1)
                nn.init.constant_(m.bias, val=0)

    def forward(self, t, y):
        return self.net(y)


class RunningAverageMeter(object):
    """Computes and stores the average and current value"""

    def __init__(self, momentum=0.99):
        self.momentum = momentum
        self.reset()

    def reset(self):
        self.val = None
        self.avg = 0

    def update(self, val):
        if self.val is None:
            self.avg = val
        else:
            self.avg = self.avg * self.momentum + val * (1 - self.momentum)
        self.val = val


if __name__ == '__main__':

    ii = 0

    func = ODEFunc().to(device)
    
    optimizer = optim.RMSprop(func.parameters(), lr=1e-3)
    end = time.time()

    time_meter = RunningAverageMeter(0.97)
    
    loss_meter = RunningAverageMeter(0.97)

    for itr in range(1, args.niters + 1):
        optimizer.zero_grad()

        # batch_y0, batch_t, batch_y = get_batch()
        # pred_y = odeint(func, batch_y0, batch_t).to(device)
        # loss = torch.mean(torch.abs(pred_y - batch_y))

        pred_y = odeint(func, true_y0, t).to(device)
        loss = torch.mean(torch.abs(pred_y - true_y))

        loss.backward()
        optimizer.step()

        time_meter.update(time.time() - end)
        loss_meter.update(loss.item())

        if itr % args.test_freq == 0:
            with torch.no_grad():
                pred_y = odeint(func, true_y0, t)
                loss = torch.mean(torch.abs(pred_y - true_y))
                print('Iter {:04d} | Total Loss {:.6f}'.format(itr, loss.item()))
                visualize(true_y, pred_y, func, ii)
                ii += 1

        end = time.time()

In [ ]:
# Record the start time
start_time = datetime.now()
print(f"Run started at: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

# import and navigate through the dataset
time = np.load('time.npy')
output = np.load('ave_tensor.npy')

strain_full = np.zeros((output.shape[0], output.shape[1], 6))
stress_full = np.zeros((output.shape[0], output.shape[1], 6))

# Fill the strain tensor components
strain_full[:, :, 0] = output[:, :, 0]  # E11
strain_full[:, :, 1] = output[:, :, 1]  # E22
strain_full[:, :, 2] = output[:, :, 2]  # E33
strain_full[:, :, 3] = output[:, :, 3]  # E12 
strain_full[:, :, 4] = output[:, :, 4]  # E13
strain_full[:, :, 5] = output[:, :, 5]  # E23

# Fill the stress tensor components
stress_full[:, :, 0] = output[:, :, 6]   # S11
stress_full[:, :, 1] = output[:, :, 7]   # S22
stress_full[:, :, 2] = output[:, :, 8]   # S33
stress_full[:, :, 3] = output[:, :, 9]   # S12 
stress_full[:, :, 4] = output[:, :, 10]  # S13
stress_full[:, :, 5] = output[:, :, 11]  # S23

# Record the end time
end_time = datetime.now()
print(f"Run ended at: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")

# Optionally, print the duration
duration = end_time - start_time
print(f"Total run duration: {duration}")

In [ ]:
# Define the split percentages (60% training, 20% validation, 20% testing)
train_split = 0.6

# Number of samples
num_trajectory = strain_full.shape[0]

val_split = (1-train_split)/2
test_split = 1 - train_split - val_split


# Compute the split indices
train_idx = math.floor(num_trajectory * train_split)
val_idx = math.floor(num_trajectory * (train_split + val_split))

# F_full = torch.tensor(F_full, dtype=torch.float32)
strain_full = torch.tensor(strain_full, dtype=torch.float32)
stress_full = torch.tensor(stress_full, dtype=torch.float32)

strain_train = strain_full[:train_idx,:,:]
strain_val = strain_full[train_idx:val_idx,:,:]
strain_test = strain_full[val_idx:,:,:]

stress_train = stress_full[:train_idx,:,:]
stress_val = stress_full[train_idx:val_idx,:,:]
stress_test = stress_full[val_idx:,:,:]


def custom_scale(data):
    max_val = np.max(abs(data))
    scaled_data = data / max_val
    return scaled_data

# Scale the data
# scaler_E = MinMaxScaler(feature_range=(-1, 1), copy=True, clip=True)
# scaler_S = MinMaxScaler(feature_range=(-1, 1), copy=True, clip=True)

scaler_E = MaxAbsScaler()
scaler_S = MaxAbsScaler()

# strain_train_reshaped = strain_train.view(-1, strain_train.size(-1)).numpy()  # Reshape to 2D for scaling
# stress_train_reshaped = stress_train.view(-1, stress_train.size(-1)).numpy()
strain_train_reshaped=strain_train.reshape(-1, strain_train.size(-1))
stress_train_reshaped=stress_train.reshape(-1, stress_train.size(-1))
num_datasets=np.size(strain_train,0)
timesteps=np.size(strain_train,1)
strain_train_scaled = torch.tensor(scaler_E.fit_transform(strain_train_reshaped).reshape(num_datasets, timesteps, strain_train.size(-1)), dtype=torch.float32)
stress_train_scaled = torch.tensor(scaler_S.fit_transform(stress_train_reshaped).reshape(num_datasets, timesteps, stress_train.size(-1)), dtype=torch.float32)

# strain_val_reshaped = strain_val.view(-1, strain_val.size(-1)).numpy()  # Reshape to 2D for scaling
# stress_val_reshaped = stress_val.view(-1, stress_val.size(-1)).numpy()
strain_val_reshaped=strain_val.reshape(-1, strain_val.size(-1))
stress_val_reshaped=stress_val.reshape(-1, stress_val.size(-1))
num_datasets=np.size(strain_val,0)
timesteps=np.size(stress_val,1)
strain_val_scaled = torch.tensor(scaler_E.transform(strain_val_reshaped).reshape(num_datasets, timesteps, strain_val.size(-1)), dtype=torch.float32)
stress_val_scaled = torch.tensor(scaler_S.transform(stress_val_reshaped).reshape(num_datasets, timesteps, stress_val.size(-1)), dtype=torch.float32)

# strain_test_reshaped = strain_test.view(-1, strain_test.size(-1)).numpy()  # Reshape to 2D for scaling
# stress_test_reshaped = stress_test.view(-1, stress_test.size(-1)).numpy()
strain_test_reshaped=strain_test.reshape(-1, strain_test.size(-1))
stress_test_reshaped=stress_test.reshape(-1, stress_test.size(-1))
num_datasets=np.size(strain_test,0)
timesteps=np.size(strain_test,1)
strain_test_scaled = torch.tensor(scaler_E.transform(strain_test_reshaped).reshape(num_datasets, timesteps, strain_test.size(-1)), dtype=torch.float32)
stress_test_scaled = torch.tensor(scaler_S.transform(stress_test_reshaped).reshape(num_datasets, timesteps, stress_test.size(-1)), dtype=torch.float32)

# # Extract scaling parameters
# print(scaler_E.min_)
# scaling_params_strain = {
#     "min_": scaler_E.min_,           # min_ is used to scale data
#     "scale_": scaler_E.scale_,       # scale_ is the scaling factor
#     "data_min": scaler_E.data_min_, # original data min
#     "data_max": scaler_E.data_max_, # original data max
#     "data_range": scaler_E.data_range_ # data range (max - min)
# }

# # Save the scaling parameters to a .dat file
# with open('scaling_parameters_strain.dat', 'w') as file:
#     for key, value in scaling_params_strain.items():
#         file.write(f"{key}: {value.tolist()}\n")


# # Extract scaling parameters
# print(scaler_S.min_)
# scaling_params_stress = {
#     "min_": scaler_S.min_,           # min_ is used to scale data
#     "scale_": scaler_S.scale_,       # scale_ is the scaling factor
#     "data_min": scaler_S.data_min_, # original data min
#     "data_max": scaler_S.data_max_, # original data max
#     "data_range": scaler_S.data_range_ # data range (max - min)
# }

# # Save the scaling parameters to a .dat file
# with open('scaling_parameters_stress.dat', 'w') as file:
#     for key, value in scaling_params_stress.items():
#         file.write(f"{key}: {value.tolist()}\n")

# # see the scaling parameters work
# # to scale the data: data * scale + min
# # to reverse scale the data: (data - min) / scale

In [6]:
import torch
import torch.nn as nn
import torchode as to
import torchdyn
from torchdyn.core import NeuralODE
from typing import Tuple

class SymmetricMatrix(nn.Module):
    """Custom layer to ensure output maintains symmetric matrix structure
    dim: dimension of output stress components
    """
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        # Number of unique elements in symmetric matrix
        self.n_elements = (dim * (dim + 1)) // 2
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size = x.shape[0]
        # Initialize output matrix
        output = torch.zeros(batch_size, self.dim, self.dim, device=x.device)
        
        # Fill upper triangle
        idx = 0
        for i in range(self.dim):
            for j in range(i, self.dim):
                output[:, i, j] = x[:, idx]
                output[:, j, i] = x[:, idx]  # Mirror to lower triangle
                idx += 1
        
        return output
    
class SELU(nn.Module):
    def forward(self, input):
        alpha = 1.6732632423543772848170429916717
        scale = 1.0507009873554804934193349852946
        return scale * (torch.where(input >= 0, input, alpha * (torch.exp(input) - 1)))

class FeedForwardNetwork(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=100, num_layers=2, dropout=0.0, use_batch_norm=False):
        super(FeedForwardNetwork, self).__init__()
        layers = []

        if num_layers < 1:
            # Directly connect input to output without hidden layers
            layers.append(nn.Linear(input_dim, output_dim))
        else:
            # Input layer
            layers.append(nn.Linear(input_dim, hidden_dim))
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(SELU())
            if dropout > 0.0:
                layers.append(nn.Dropout(dropout))

            # Hidden layers
            for _ in range(num_layers):
                layers.append(nn.Linear(hidden_dim, hidden_dim))
                if use_batch_norm:
                    layers.append(nn.BatchNorm1d(hidden_dim))
                layers.append(SELU())
                if dropout > 0.0:
                    layers.append(nn.Dropout(dropout))

            # Output layer
            layers.append(nn.Linear(hidden_dim, output_dim))

        # Combine all layers into a Sequential module
        self.layers = nn.Sequential(*layers)        

    def forward(self, x):
        x = self.layers(x)
        return x 
    

class NO(nn.Module):
    """Neural Operator for material deformation"""
    def __init__(self, stress_dim: int = 6, k_isv: int = 3, hidden_dim: int = 64, num_layers: int = 3):
    # def __init__(self, stress_dim, k_isv, hidden_dim, num_layers):
        super().__init__()

        self.stress_dim = stress_dim
        self.k_isv = k_isv
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.NN = FeedForwardNetwork(stress_dim + k_isv, int((stress_dim * (stress_dim+1)/2) + (stress_dim * k_isv)) , hidden_dim, num_layers)
        
        self.symmetric_layer = SymmetricMatrix(stress_dim)
    # Main body of Neural operator model
    def forward(self, e: torch.Tensor, y: torch.Tensor, solv) -> torch.Tensor:
        batch_size = e.shape[0]
        increments = e.shape[1]
        e_incr = torch.cat([torch.diff(e, dim=1), torch.zeros(batch_size, 1, e.shape[2])], dim=1) 
        
        # Initialise outputs
        stress = torch.zeros(batch_size, increments, self.stress_dim, device=e.device)
        isv = torch.zeros(batch_size, increments, self.k_isv, device=e.device)
        step = torch.linspace(0, 1, 2)

        for n in range(increments-1):
            e_incr_n = e_incr[:, n]

            # Get neural network output
            nn_output = self.NN(y)

            # Split output into stress and internal variables derivatives
            d_stress = nn_output[:, :int(self.stress_dim * (self.stress_dim+1)/2)]  # First stress_dim elements for symmetric matrix
            d_isv = nn_output[:, int(self.stress_dim * (self.stress_dim+1)/2):]  # Remaining elements for isv

            # Reshape stress derivatives into symmetric matrix
            d_stress_matrix = self.symmetric_layer(d_stress)

            # Reshape the tensor
            d_isv_matrix = d_isv.view(batch_size, self.k_isv, self.stress_dim)
            # print(d_stress_matrix.shape)
            # print(d_isv_matrix.shape)
            Jac= torch.cat((d_stress_matrix, d_isv_matrix), dim=1)
            # print(Jac.shape)
            # print(e_incr_n.shape)
            y = torch.bmm(Jac, e_incr_n.unsqueeze(2))      # Shape: (Batch, 1, stress_dim + k_isv)
            y = y.squeeze(2)                               # Shape: (Batch, stress_dim + k_isv)
            
            stress[:, n+1] = stress[:, n] + y[:, :self.stress_dim]
            isv[:, n+1] = isv[:, n] + y[:, self.stress_dim:]
        
        return stress, isv

def Learner(no, solv, num_epochs, batch_size, stress_dim, k_isv, learning_rate, w_decay_value):
    optimizer = optim.Adam(no.parameters(), lr=learning_rate, weight_decay = w_decay_value)
    loss_MSE = nn.MSELoss()     # loss_normalised
    loss_L1 = nn.L1Loss()       # loss_normalised
    
    train_losses = []
    train_lossesL2 = []
    val_losses = []
    val_lossesL2 = []

    for epoch in range(num_epochs):
        epoch_train_loss = 0.0  # To track loss per epoch
        epoch_val_loss = 0.0  # To track loss per epoch
        epoch_train_lossL2 = 0.0  # To track loss per epoch
        epoch_val_lossL2 = 0.0  # To track loss per epoch
        
        batch_num = len(range(0, strain_train_scaled.size(0), batch_size))
        
        for i in range(0, strain_train_scaled.size(0), batch_size):
            F_train_batch = strain_train_scaled[i:i + batch_size,:,:]            
            xi_train_initial = torch.zeros(F_train_batch.size(0), stress_dim + k_isv)
            stress_train_batch = stress_train_scaled[i:i + batch_size,:,:]
            dstress_train_batch = torch.cat([torch.diff(stress_train_batch, dim=1), torch.zeros(stress_train_batch.shape[0], 1, stress_train_batch.shape[2])], dim=1) 

            F_val_batch = strain_val_scaled            
            xi_val_initial = torch.zeros(F_val_batch.size(0), stress_dim + k_isv)
            stress_val_batch = stress_val_scaled
            dstress_val_batch = torch.cat([torch.diff(stress_val_batch, dim=1), torch.zeros(stress_val_batch.shape[0], 1, stress_val_batch.shape[2])], dim=1) 

            # Forward pass
            predicted_stress_train, predicted_isv_train = no(F_train_batch, xi_train_initial, solv)
            with torch.no_grad():  # Disable gradient calculation
                predicted_stress_val,  predicted_isv_val = no(F_val_batch, xi_val_initial, solv)
            
            # Compute loss
            loss_train = loss_MSE(predicted_stress_train, stress_train_batch)#loss_MSE(predicted_stress_train, stress_train_batch)
            loss_val = loss_MSE(predicted_stress_val, stress_val_batch)
            
            loss_trainL2 = loss_MSE(predicted_stress_train, stress_train_batch)
            loss_valL2 = loss_MSE(predicted_stress_val, stress_val_batch)

            # Backward pass and optimize
            optimizer.zero_grad()
            loss_train.backward()
            optimizer.step()

            epoch_train_loss += loss_train.item()/batch_num
            epoch_val_loss += loss_val.item()/batch_num
            epoch_train_lossL2 += loss_trainL2.item()/batch_num
            epoch_val_lossL2 += loss_valL2.item()/batch_num
        
        train_losses.append(epoch_train_loss)
        val_losses.append(epoch_val_loss)
        train_lossesL2.append(epoch_train_lossL2)
        val_lossesL2.append(epoch_val_lossL2)

        if (epoch+1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Training Loss: {epoch_train_lossL2:.4f}, Validation Loss: {epoch_val_lossL2:.4f}')
    
    return train_losses, val_losses, train_lossesL2, val_lossesL2
    
    # def forward(self, t: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    #     batch_size = y.shape[0]
        
    #     # Split input into stress and internal variables
    #     stress = y[:, :self.stress_dim]
    #     h = y[:, self.stress_dim:]
        
    #     # Get neural network output
    #     nn_output = self.network(y)
        
    #     # Split output into stress and internal variables derivatives
    #     stress_deriv = nn_output[:, :21]  # First 21 elements for symmetric matrix
    #     h_deriv = nn_output[:, 21:]  # Remaining elements for internal variables
        
    #     # Reshape stress derivatives into symmetric matrix
    #     stress_matrix = self.symmetric_layer(stress_deriv)
        
    #     # Flatten stress matrix and concatenate with internal variables
    #     output = torch.cat([stress_matrix.reshape(batch_size, -1), h_deriv], dim=1)
    #     print(output.shape)
    #     return output

# class VonMisesModel:
#     """Wrapper class for the complete Von Mises model"""
#     def __init__(self, hidden_dim: int = 64, k_states: int = 3):
#         self.neural_operator = NO(hidden_dim, k_states)
#         # self.neural_ode = NeuralODE(
#         #     self.neural_operator,
#         #     sensitivity='adjoint',
#         #     solver='dopri5'
#         # )
#         self.term = to.ODETerm(self.neural_operator)
#         self.step_method = to.Dopri5(term=self.term)
#         self.step_size_controller = to.IntegralController(atol=1e-6, rtol=1e-3, term=self.term)
#         self.adjoint = to.AutoDiffAdjoint(self.step_method, self.step_size_controller).to(device)

#     def train_step(self, 
#                   optimizer: torch.optim.Optimizer,
#                   batch: Tuple[torch.Tensor, torch.Tensor]) -> float:
#         """Perform single training step"""
        
#         x, y_true = batch
#         optimizer.zero_grad()
#         print(x.shape)
#         print(y_true.shape)

#         batch_size = 3
#         t_eval = torch.tile(torch.linspace(0.0, 1.0, 10), (batch_size, 1))
#         problem = to.InitialValueProblem(y0=torch.zeros((batch_size, 5)).to(device), t_eval=t_eval.to(device))
#         sol = self.adjoint.solve(problem)
#         y_pred = sol.ys
#         # Forward pass
#         # y_pred = self.neural_ode(x)
        
#         # Compute loss (MSE)
#         loss = torch.mean((y_pred - y_true) ** 2)
        
#         # Backward pass
#         loss.backward()
#         optimizer.step()
        
#         return loss.item()

# Example usage
if __name__ == "__main__":
    # # Initialize model
    # model = VonMisesModel(hidden_dim=64, k_states=3)
    
    # # Initialize optimizer
    # optimizer = torch.optim.Adam(model.neural_operator.parameters(), lr=1e-3)
    
    # # Training loop (with dummy data for illustration)
    # batch_size = 32
    # input_dim = 9  # 6 stress components + 3 internal variables
    
    # for epoch in range(100):
    #     # Generate dummy training data
    #     x = torch.randn(batch_size, input_dim)
    #     y = torch.randn(batch_size, input_dim)  # Target values
        
    #     # Training step
    #     loss = model.train_step(optimizer, (x, y))
        
    #     if epoch % 10 == 0:
    #         print(f"Epoch {epoch}, Loss: {loss:.6f}")

    # ML model and training
    # Set hyperparameters
    batch_size = strain_train_scaled.size(0)
    num_epochs = 100
    learning_rate = 0.001
    w_decay_value= 0
    solv = "euler"  # Example: it could also be "NeuralODE"


    # Inputs
    d = 1  # Dimension of strain or stress componants
    # for k in [1,4]:  # Run k from 1 to 10
    num_layers=2
    hidden_dim=64
    k=3


    # To store results
    results = {}

    # Create RNO model
    no = NO(d, k, hidden_dim, num_layers)

    # Train the model
    train_losses,  val_losses, train_lossesL2,  val_lossesL2 = Learner(no, solv, num_epochs, batch_size, d, k, learning_rate, w_decay_value)

    # Store results in dictionary
    result = {
        "num_epochs": num_epochs,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "train_lossesL2": train_lossesL2,
        "val_lossesL2": val_lossesL2
    }

    # Saving the trained model
    torch.save(no, 'NO_model_k{}_L{}_hd{}_epoch{}_batch{}_trainSize{}.pth'.format(k,num_layers,hidden_dim,num_epochs,batch_size,train_idx))

    # Save the results to a file
    np.save('NO_training_results_k{}_L{}_hd{}_epoch{}_batch{}_trainSize{}.npy'.format(k,num_layers,hidden_dim,num_epochs,batch_size,train_idx), results)



Epoch [10/100], Training Loss: 0.1515, Validation Loss: 0.1431
Epoch [20/100], Training Loss: 0.1479, Validation Loss: 0.1392
Epoch [30/100], Training Loss: 0.1458, Validation Loss: 0.1454
Epoch [40/100], Training Loss: 0.1432, Validation Loss: 0.1580
Epoch [50/100], Training Loss: 0.1373, Validation Loss: 0.2392
Epoch [60/100], Training Loss: 0.1282, Validation Loss: 0.8282
Epoch [70/100], Training Loss: 0.1283, Validation Loss: 1.3409
Epoch [80/100], Training Loss: 0.1280, Validation Loss: 0.8775
Epoch [90/100], Training Loss: 0.1277, Validation Loss: 1.0797
Epoch [100/100], Training Loss: 0.1277, Validation Loss: 1.1163


In [7]:
# plots

plt.figure(figsize=(5, 4))
# for k in [1,1]:

plt.plot(range(result['num_epochs']), result['train_losses'] , label='Training Loss', color='blue')
plt.plot(range(result['num_epochs']), result['val_losses'] , label='Validation Loss', color='red')
# plt.yscale('log')
plt.xlabel('Epoch', fontsize=14, fontname='Times New Roman')
plt.ylabel('MSE Loss', fontsize=14, fontname='Times New Roman')
plt.tick_params(axis='both', which='major', direction='in')
plt.tick_params(axis='both', which='minor', direction='in')
# plt.autoscale(tight=True)
plt.legend(prop={'family': 'Times New Roman', 'size': 14}, frameon=False)
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 4))
plt.plot(range(result['num_epochs']), result['train_lossesL2']  , label='Training Loss', color='blue')
plt.plot(range(result['num_epochs']), result['val_lossesL2']  , label='Validation Loss', color='red')
#plt.xscale('log')
plt.yscale('log')
plt.xlabel('Epoch', fontsize=14, fontname='Times New Roman')
plt.ylabel('MSE Loss', fontsize=14, fontname='Times New Roman')
plt.tick_params(axis='both', which='major', direction='in')
plt.tick_params(axis='both', which='minor', direction='in')
# plt.autoscale(tight=True)
plt.legend(prop={'family': 'Times New Roman', 'size': 14}, frameon=False)
plt.tight_layout()
plt.show()

In [8]:
# prediction using the trained model
dt=time[0,1,0]-time[0,0,0]
xi_initial_test = torch.zeros(strain_val_scaled.size(0), no.stress_dim + no.k_isv)
val_stress_scaled, val_xi_history_scaled = no(strain_val_scaled, xi_initial_test,solv)
# predicted_stress = scaler_S.inverse_transform(predicted_stress_scaled)
val_stress = scaler_S.inverse_transform(val_stress_scaled.view(-1, val_stress_scaled.size(-1)).detach().numpy())
num_datasets=np.size(val_stress_scaled,0)
timesteps=np.size(val_stress_scaled,1)
val_stress = torch.tensor(val_stress.reshape(num_datasets, timesteps, val_stress_scaled.size(-1)), dtype=torch.float32)

print(f"Predicted stress tensor shape: {val_stress.shape}")

# Calculate absolute residuals on the calibration set
PI=0.99
residuals = torch.abs(stress_val - val_stress)
print(f"residuals shape: {residuals.shape}")
q_hat = torch.quantile(residuals, PI)  # 95% prediction interval
print(f"q_hat: {q_hat}")

Predicted stress tensor shape: torch.Size([499, 200, 1])
residuals shape: torch.Size([499, 200, 1])
q_hat: 1286471296.0


In [9]:
# prediction using the trained model
dt = time[0,1,0] - time[0,0,0]
xi_initial_test = torch.zeros(strain_test_scaled.size(0), no.stress_dim + no.k_isv)
predicted_stress_scaled, predicted_xi_history_scaled = no(strain_test_scaled, xi_initial_test,solv)
# predicted_stress = scaler_S.inverse_transform(predicted_stress_scaled)
predicted_stress = scaler_S.inverse_transform(predicted_stress_scaled.view(-1, predicted_stress_scaled.size(-1)).detach().numpy())
predicted_xi_history = scaler_E.inverse_transform(predicted_xi_history_scaled.view(-1, predicted_xi_history_scaled.size(-1)).detach().numpy())
num_datasets=np.size(predicted_stress_scaled,0)
timesteps=np.size(predicted_stress_scaled,1)
predicted_stress = torch.tensor(predicted_stress.reshape(num_datasets, timesteps, predicted_stress_scaled.size(-1)), dtype=torch.float32)
predicted_xi_history = torch.tensor(predicted_xi_history.reshape(num_datasets, timesteps, predicted_xi_history_scaled.size(-1)), dtype=torch.float32)
lower_bound = predicted_stress - q_hat
upper_bound = predicted_stress + q_hat
print(f"Predicted stress tensor shape: {predicted_stress.shape}")
print(f"upper_bound stress tensor shape: {upper_bound.shape}")

Predicted stress tensor shape: torch.Size([500, 200, 1])
upper_bound stress tensor shape: torch.Size([500, 200, 1])


In [10]:
traj_ID = 15                               # Trajectory ID to visualize
mag_scale = 1e6

plt.rcParams['mathtext.fontset'] = 'stix'  # Use 'stix' or 'cm' for fonts that match Times New Roman
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']

plt.figure(figsize=(5, 4))
plt.plot(strain_test[traj_ID, :, 0].numpy(), stress_test[traj_ID, :, 0].numpy()/mag_scale, 'k.', label=r'test set ({})'.format(traj_ID))
plt.plot(strain_test[traj_ID, :, 0].numpy(), predicted_stress[traj_ID, :, 0].detach().numpy()/mag_scale, 'r', label='model estimation')
# plt.fill_between(strain_test[traj_ID, :, 0].numpy(),lower_bound[traj_ID, :, 0].numpy()/mag_scale, upper_bound[traj_ID, :, 0].numpy()/mag_scale,color='blue', alpha=0.2, label=r'CI ({}%)'.format(PI*100))
Xlim=np.max(np.abs(strain_test[traj_ID, :, 0].numpy())) + np.max(np.abs(strain_test[traj_ID, :, 0].numpy())) * 0.1
plt.xlim(-Xlim, Xlim)
Ylim=np.max(np.abs(stress_test[traj_ID, :, 0].numpy()/mag_scale)) + np.max(np.abs(stress_test[traj_ID, :, 0].numpy()/mag_scale)) * 0.1
plt.ylim(-Ylim, Ylim)
plt.xlabel('$\epsilon$', fontsize=18)
plt.ylabel('$\sigma~(MPa)$', fontsize=18)  # , fontname='Times New Roman'
plt.legend(loc='upper left', prop={'family': 'Times New Roman', 'size': 12})
plt.grid(True, linestyle='--')
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 4))
plt.plot(strain_full[traj_ID+800, :, 0], state[traj_ID+800, :, :], 'k.', label='reference ISV')
plt.plot(strain_test[traj_ID, :, :].numpy(), predicted_xi_history[traj_ID, :, :].detach().numpy(), 'r', label='predicted ISV')
Xlim=torch.max(torch.abs(strain_full[traj_ID+800, :, 0])) + torch.max(torch.abs(strain_full[traj_ID+800, :, 0])) * 0.1
plt.xlim(-Xlim, Xlim)
Ylim=np.max(np.abs(predicted_xi_history[traj_ID, :, :].detach().numpy())) + np.max(np.abs(predicted_xi_history[traj_ID, :, :].detach().numpy())) * 0.1
plt.ylim(-Ylim, Ylim)
plt.xlabel('$\epsilon$', fontsize=18)
plt.ylabel('$ISV~()$', fontsize=18)  # , fontname='Times New Roman'
plt.legend(loc='upper right', prop={'family': 'Times New Roman', 'size': 12})
plt.grid(True, linestyle='--')
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 4))
plt.plot(state[traj_ID+800, :, :], 'k.', label='reference ISV')
plt.plot(predicted_xi_history[traj_ID, :, :].detach().numpy(), 'r', label='predicted ISV')
Ylim=np.max(np.abs(predicted_xi_history[traj_ID, :, :].detach().numpy())) + np.max(np.abs(predicted_xi_history[traj_ID, :, :].detach().numpy()))*0.1
plt.ylim(-Ylim, Ylim)
plt.xlabel('Increment', fontsize=18)
plt.ylabel('$ISV~()$', fontsize=18) # , fontname='Times New Roman'
plt.legend(loc='upper right', prop={'family': 'Times New Roman', 'size': 12})
plt.grid(True, linestyle='--')
plt.tight_layout()
plt.show()

# predicted_xi_history_sum=torch.sum(predicted_xi_history_scaled[traj_ID, :, :], axis=-1)
# plt.figure(figsize=(5, 4))
# plt.plot(strain_test[traj_ID, :, 0].numpy(), predicted_xi_history_sum.detach().numpy(), label='sum')
# plt.plot(strain_full[traj_ID+800, :, 0], state[traj_ID+800, :, :], label='k')
# Ylim=torch.max(torch.abs(state[traj_ID+800, :, :])) + torch.max(torch.abs(state[traj_ID+800, :, :]))*0.1
# plt.ylim(-Ylim, Ylim)
# xlim=np.max(np.abs(strain_full[traj_ID+800, :, 0])) + np.max(np.abs(strain_full[traj_ID+800, :, 0]))*0.1
# plt.xlim(-xlim, xlim)
# plt.xlabel('$\epsilon$', fontsize=18)
# plt.ylabel('$ISV~()$', fontsize=18)# , fontname='Times New Roman'
# plt.legend(loc='upper left', prop={'family': 'Times New Roman', 'size': 12})
# plt.grid(True, linestyle='--')
# plt.tight_layout()
# plt.show()

<>:16: SyntaxWarning: invalid escape sequence '\e'
<>:17: SyntaxWarning: invalid escape sequence '\s'
<>:30: SyntaxWarning: invalid escape sequence '\e'
<>:16: SyntaxWarning: invalid escape sequence '\e'
<>:17: SyntaxWarning: invalid escape sequence '\s'
<>:30: SyntaxWarning: invalid escape sequence '\e'
C:\Users\lt24550\AppData\Local\Temp\ipykernel_36020\2558006833.py:16: SyntaxWarning: invalid escape sequence '\e'
  plt.xlabel('$\epsilon$', fontsize=18)
C:\Users\lt24550\AppData\Local\Temp\ipykernel_36020\2558006833.py:17: SyntaxWarning: invalid escape sequence '\s'
  plt.ylabel('$\sigma~(MPa)$', fontsize=18)  # , fontname='Times New Roman'
C:\Users\lt24550\AppData\Local\Temp\ipykernel_36020\2558006833.py:30: SyntaxWarning: invalid escape sequence '\e'
  plt.xlabel('$\epsilon$', fontsize=18)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchdyn.core import NeuralODE
import numpy as np
import matplotlib.pyplot as plt

class SymmetricMatrix(nn.Module):
    """Custom layer to ensure output maintains symmetric matrix structure"""
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        self.n_elements = (dim * (dim + 1)) // 2
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size = x.shape[0]
        output = torch.zeros(batch_size, self.dim, self.dim, device=x.device)
        
        idx = 0
        for i in range(self.dim):
            for j in range(i, self.dim):
                output[:, i, j] = x[:, idx]
                output[:, j, i] = x[:, idx]
                idx += 1
        
        return output

class VonMisesNeuralOperator(nn.Module):
    def __init__(self, hidden_dim: int = 64, k_states: int = 3):
        super().__init__()
        self.k_states = k_states
        
        # Dimensions for the symmetric stress tensor (6 components for 3D)
        self.stress_dim = 6
        self.total_dim = self.stress_dim + k_states
        
        # Neural network for the operator
        self.net = nn.Sequential(
            nn.Linear(self.total_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 21 + 6 * k_states)
        )
        
        self.symmetric_layer = SymmetricMatrix(6)
    
    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the neural operator.
        Args:
            x: Input tensor of shape (batch_size, total_dim)
            t: Time tensor (not used in this implementation but required by torchdyn)
        Returns:
            torch.Tensor: Output derivatives
        """
        # Get neural network output
        nn_output = self.net(x)
        
        # Split output into stress and internal variables derivatives
        stress_deriv = nn_output[:, :21]  # First 21 elements for symmetric matrix
        h_deriv = nn_output[:, 21:]  # Remaining elements for internal variables
        
        # Convert stress derivatives to symmetric matrix
        stress_matrix = self.symmetric_layer(stress_deriv)
        
        # Flatten stress matrix and concatenate with internal variables
        stress_flat = stress_matrix.reshape(x.shape[0], -1)[:, :6]  # Take only the needed 6 components
        output = torch.cat([stress_flat, h_deriv], dim=1)
        
        return output

def get_batch(stress_trajectories, strain_trajectories, batch_size, batch_time, data_size):
    """Create a batch of trajectories"""
    # Random starting points
    s = torch.from_numpy(
        np.random.choice(np.arange(data_size - batch_time, dtype=np.int64), 
                        batch_size, replace=False)
    )
    
    # Extract sequences
    batch_y0 = torch.cat([
        stress_trajectories[s, :],  # Initial stress
        strain_trajectories[s, :]   # Initial strain
    ], dim=1)
    
    # Create time points
    batch_t = torch.linspace(0, batch_time-1, batch_time)
    
    # Extract trajectory sequences
    batch_y = torch.stack([
        torch.cat([
            stress_trajectories[s + i, :],
            strain_trajectories[s + i, :]
        ], dim=1) for i in range(batch_time)
    ], dim=0)
    
    return batch_y0, batch_t, batch_y

def train_model(stress_trajectories, strain_trajectories, hidden_dim=64, k_states=3, 
                batch_size=20, batch_time=10, n_epochs=100, device='cpu'):
    """Train the Von Mises neural operator model"""
    # Convert data to torch and move to device
    stress_trajectories = torch.FloatTensor(stress_trajectories).to(device)
    strain_trajectories = torch.FloatTensor(strain_trajectories).to(device)
    
    # Initialize model
    func = VonMisesNeuralOperator(hidden_dim=hidden_dim, k_states=k_states).to(device)
    
    # Create NeuralODE with the correct configuration
    neural_ode = NeuralODE(
        func,
        sensitivity='adjoint',
        solver='dopri5',
        atol=1e-4,
        rtol=1e-4
    ).to(device)
    
    optimizer = optim.Adam(func.parameters(), lr=1e-3)
    
    data_size = len(stress_trajectories)
    losses = []
    
    for epoch in range(n_epochs):
        optimizer.zero_grad()
        
        # Get batch
        batch_y0, batch_t, batch_y = get_batch(
            stress_trajectories, strain_trajectories, 
            batch_size, batch_time, data_size
        )
        
        # Forward pass
        pred_y = neural_ode.forward(batch_y0, batch_t)
        
        # Compute loss
        loss = torch.mean((pred_y - batch_y) ** 2)
        losses.append(loss.item())
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        if epoch % 10 == 0:
            print(f'Epoch {epoch}, Loss: {loss.item():.6f}')
    
    return func, losses

# Example usage
if __name__ == "__main__":
    # Generate dummy data
    n_trajectories = 1000
    n_timesteps = 101
    n_components = 6
    
    # Create dummy stress and strain trajectories
    stress_trajectories = np.random.randn(n_timesteps, n_trajectories, n_components)
    strain_trajectories = np.random.randn(n_timesteps, n_trajectories, n_components)
    
    # Train model
    model, losses = train_model(
        stress_trajectories, 
        strain_trajectories,
        hidden_dim=64,
        k_states=3,
        batch_size=20,
        batch_time=10,
        n_epochs=100
    )
    
    # Plot losses
    plt.figure(figsize=(10, 5))
    plt.plot(losses)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Loss')
    plt.yscale('log')
    plt.show()

In [ ]:

class SymmetricMatrix(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        self.n_elements = (dim * (dim + 1)) // 2
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size = x.shape[0]
        output = torch.zeros(batch_size, self.dim, self.dim, device=x.device)
        
        idx = 0
        for i in range(self.dim):
            for j in range(i, self.dim):
                output[:, i, j] = x[:, idx]
                output[:, j, i] = x[:, idx]
                idx += 1
        
        return output

class ConstitutiveODEFunc(nn.Module):
    def __init__(self, k_states: int = 3, hidden_dim: int = 50):
        super().__init__()
        self.k_states = k_states
        
        # Input dimension: 6 (stress) + k (internal variables)
        input_dim = 6 + k_states
        
        # Output dimension: 21 (symmetric 6x6 matrix) + 6*k (internal variables jacobian)
        output_dim = 21 + 6 * k_states
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim)
        )
        
        self.symmetric_layer = SymmetricMatrix(6)
        
        # Initialize weights
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0, std=0.1)
                nn.init.constant_(m.bias, val=0)
    
    def forward(self, t, y):
        # Split input into stress and internal variables
        stress = y[..., :6]
        h = y[..., 6:]
        
        # Get network output
        network_output = self.net(y)
        
        # Split output into stress and internal variables derivatives
        stress_deriv = network_output[..., :21]
        h_deriv = network_output[..., 21:]
        
        # Convert stress derivatives to symmetric matrix and flatten
        stress_matrix = self.symmetric_layer(stress_deriv)
        stress_matrix_flat = stress_matrix.reshape(stress_matrix.shape[0], -1)
        
        # Concatenate with internal variables derivatives
        return torch.cat([stress_matrix_flat[..., :6], h_deriv], dim=-1)

class RunningAverageMeter(object):
    def __init__(self, momentum=0.99):
        self.momentum = momentum
        self.reset()

    def reset(self):
        self.val = None
        self.avg = 0

    def update(self, val):
        if self.val is None:
            self.avg = val
        else:
            self.avg = self.avg * self.momentum + val * (1 - self.momentum)
        self.val = val

def visualize(true_y, pred_y, t, itr):
    """Visualize the first component of stress and internal variables"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Plot stress components
    ax1.set_title('Stress Components')
    ax1.plot(t.cpu().numpy(), true_y.cpu().numpy()[:, 0, :6], 'g-', alpha=0.3, label='True')
    ax1.plot(t.cpu().numpy(), pred_y.cpu().numpy()[:, 0, :6], 'b--', alpha=0.3, label='Predicted')
    ax1.set_xlabel('t')
    ax1.set_ylabel('Stress')
    ax1.legend()
    
    # Plot internal variables
    ax2.set_title('Internal Variables')
    ax2.plot(t.cpu().numpy(), true_y.cpu().numpy()[:, 0, 6:], 'g-', alpha=0.3, label='True')
    ax2.plot(t.cpu().numpy(), pred_y.cpu().numpy()[:, 0, 6:], 'b--', alpha=0.3, label='Predicted')
    ax2.set_xlabel('t')
    ax2.set_ylabel('Internal Variables')
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig(f'constitutive_model_{itr:03d}.png')
    plt.close()

def train_model(true_y0, t, true_y, k_states=3, niters=2000, batch_time=10, batch_size=20, test_freq=20, device='cuda'):
    func = ConstitutiveODEFunc(k_states=k_states).to(device)
    optimizer = optim.RMSprop(func.parameters(), lr=1e-3)
    
    time_meter = RunningAverageMeter(0.97)
    loss_meter = RunningAverageMeter(0.97)
    
    batch_y0 = true_y0.to(device)
    batch_t = t[:batch_time].to(device)
    
    for itr in range(1, niters + 1):
        optimizer.zero_grad()
        
        # Forward pass
        pred_y = odeint(func, batch_y0, batch_t).to(device)
        
        # Compute loss
        loss = torch.mean(torch.abs(pred_y - true_y[:batch_time]))
        loss.backward()
        optimizer.step()
        
        time_meter.update(time.time() - end)
        loss_meter.update(loss.item())
        
        if itr % test_freq == 0:
            with torch.no_grad():
                pred_y = odeint(func, true_y0, t)
                loss = torch.mean(torch.abs(pred_y - true_y))
                print(f'Iter {itr:04d} | Total Loss {loss.item():.6f}')
                visualize(true_y, pred_y, t, itr // test_freq)
        
        end = time.time()
    
    return func

# Example usage:
if __name__ == '__main__':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Generate dummy data for demonstration
    data_size = 200
    k_states = 3
    total_dim = 6 + k_states  # 6 stress components + k internal variables
    
    # Create dummy initial conditions and true trajectory
    true_y0 = torch.randn(1, total_dim).to(device)
    t = torch.linspace(0., 25., data_size).to(device)
    
    # Generate dummy true trajectory (replace this with your actual data)
    true_y = torch.stack([true_y0 * torch.exp(-0.1 * t_) + torch.randn_like(true_y0) * 0.1 
                         for t_ in t], dim=0)
    
    # Train the model
    model = train_model(true_y0, t, true_y, k_states=k_states)